# Database Construction — Part 1

**Step 1 of 2** — Raw POI extraction and category assignment.

This notebook reads per-city POI JSON files, classifies each POI via regex patterns matched against its name and descriptions, maps the resulting type labels to a two-level taxonomy (main category → sub-category), and applies manual overrides for edge cases. The final output is a flat CSV with `main_1, main_2, sub_1..sub_4` columns, which feeds directly into Part 2.

Pipeline: `cities/*.json` → v1 (type labels) → v1 multi (main/sub) → v2 (manual main override) → v3 (manual sub override) → v4 (flattened columns).

> *This notebook is kept purely for demonstration and transparency — the pipeline has already been run and is not intended to be re-executed.*

In [ ]:
import os
import json
import re
import pandas as pd

# Each key is a POI type label; each value is a list of regex patterns
# that, if matched against the POI's combined text (name + descriptions),
# cause the POI to be tagged with that label. Patterns cover both English
# and Turkish morphology (suffixed forms, declensions).
# @title
TYPE_PATTERNS = {
    # --- Core built / culture ---
    "Museum": [
        r"\bmuseum\b",
        r"\bmüze(si|sini|sinde|leri|leri(n)?i)?\b",
    ],
    "Köy": [
        r"\bköy(ü)?\b",
    ],
    "Ev": [
        r"\bev(i|leri)?\b",
    ],
    "Köprü": [
        r"\bköpru(sü)?\b",
    ],
    "Park": [
        r"\bpark\b",
        r"\bparkı\b",
        r"\bkoru(su|sunu|sunda)?\b",
        r"\btabiat park(ı|ını|ında)?\b",
        r"\bmilli park(ı|ını|ında)?\b",
        r"\bnational park\b",
        r"\bmillet bahçe(si|sini|sinde)?\b",
        r"\bkent orman(ı|ını|ında)?\b",
    ],
    "Castle": [
        r"\bcastle\b",
        r"\bkale(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bhisar\b",
        r"\bfortress\b",
        r"\btabya(sı|sını|sında|ları|ların(i)?)?\b",
        r"\bburc(u|unu|unda|ları|ların(i)?)?\b",
    ],
    "Mosque": [
        r"\bmosque\b",
        r"\bcami(i|si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bmescit\b",
    ],
    "Church": [
        r"\bchurch\b",
        r"\bkilise(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bcathedral\b",
        r"\bkatedral(i|i)?\b",
        r"\bşapel(i|i)?\b",
        r"\bchapel\b",
        r"\bsynagogue\b",
        r"\bsinagog(u|unu|unda|ları|ların(i)?)?\b",
    ],
    "Monastery": [
        r"\bmonastery\b",
        r"\bmanastır(ı|ını|ında|ları|ların(i)?)?\b",
    ],
    "Tomb": [
        r"\btomb\b",
        r"\btürbe(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bkümbet(ler|i|leri|lerin(i)?)?\b",
        r"\bmausoleum\b",
        r"\btümülüs(ü|ü|ünde|ünü|ler|leri|lerin(i)?)?\b",
    ],
    "Cemetery": [
        r"\bcemetery\b",
        r"\bmezarlık(ı|ını|ta|ta|ında|ları|ların(i)?)?\b",
        r"\bşehitlik\b",
        r"\bnekropol(ü|ünü|ünde|ler|leri|lerin(i)?)?\b",
        r"\bnecropoli(s|s)?\b",
    ],
    "Monument": [
        r"\bmonument\b",
        r"\banıt(ı|ını|ında|ları|ların(i)?)?\b",
        r"\bstatue\b",
        r"\bheykel(i|i)?\b",
        r"\byazıt(ı|ını|ında|ları|ların(i)?)?\b",
        r"\babide(si|sini|sinde|leri|lerin(i)?)?\b",
    ],
    "Tower": [
        r"\btower\b",
        r"\bkule(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bminare(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bsaat kulesi\b",
        r"\blighthouse\b",
        r"\bfener(i|i)?\b",
    ],
    "Complex": [
        r"\bkülliye(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bkültür merkez(i|i)?\b",
        r"\bcultural center\b",
        r"\bart center\b",
        r"\bgallery\b",
        r"\bgaleri(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bfu(ar|arı)\b",
        r"\bkongre merkez(i|i)?\b",
    ],
    "Madrasa": [
        r"\bmedrese(si|sini|sinde|leri|lerin(i)?)?\b",
        r"\bmadrasa\b",
    ],
    "Caravanserai_Inn": [
        r"\bcaravanserai\b",
        r"\bhan(ı|ını|ında|lar|ları|ların(i)?)?\b",
        r"\bkervansaray(ı|ını|ında|lar|ları|ların(i)?)?\b",
        r"\btaşhan\b",
        r"\bbede(st|st)en\b",
    ],
    "Bath_Hammam": [
        r"\bhamam(ı|ını|ında|lar|ları|ların(i)?)?\b",
        r"\bbath(s)?\b",
        r"\broma hamam(ı|ını|ında)?\b",
        r"\broman bath(s)?\b",
    ],
    "Bazaar_Market": [
        r"\bçarşı(sı|sını|sında|lar|ları|ların(i)?)?\b",
        r"\bbazaar\b",
        r"\bmarket\b",
        r"\bgrand bazaar\b",
        r"\bkapalıçarşı\b",
        r"\bmısır çarşısı\b",
        r"\barasta\b",
    ],
    "Square": [
        r"\bmeydan(ı|ını|ında|lar|ları|ların(i)?)?\b",
        r"\bsquare\b",
        r"\bpublic square\b",
    ],
    "City_Walls_Gates": [
        r"\bsur(lar|ları|ların(i)?)?\b",
        r"\bwall(s)?\b",
        r"\bgate\b",
        r"\bkapı(sı|sını|sında|lar|ları|ların(i)?)?\b",
    ],
    "Cistern": [
        r"\bcistern\b",
        r"\bsarnıç(ı|ını|ta|ında|lar|ları|ların(i)?)?\b",
        r"\byerebatan\b",
    ],
    "Underground_City": [
        r"\bunderground city\b",
        r"\byer ?altı şehr(i|i)?\b",
        r"\byer ?altı şehir\b",
        r"\byeraltı şehri\b",
    ],
    "Ancient_City": [
        r"\barchaeological\b",
        r"\bantik kent(i)?\b",
        r"\bancient city\b",
        r"\bruins\b",
        r"\bharabe(leri|si|sini|sinde)?\b",
        r"\bören ?yeri\b",
        r"\börenyeri\b",
    ],

    "Burial_Site": [
        r"\b(kaya )?mezar(lığı|ları|ları(n)?ı)?\b",
        r"\bhöyük(ü|ünü|ünde|ler|leri|lerin(i)?)?\b",
        r"\bmound\b",
        r"\bdolmen\b",
        r"\btümülüs(ü|ünü|ünde|ler|leri|lerin(i)?)?\b",
    ],

    "Ancient_Structure": [
        r"\btemple\b",
        r"\btapınak(ı|ını|ta|ında|lar|ları|ların(i)?)?\b",
        r"\btheatre\b",
        r"\btiyatro(su|sunu|sunda|lar|ları|ların(i)?)?\b",
        r"\bagora\b",
        r"\bbouleuterion\b",
    ],

    "Ancient_Relief_Art": [
        r"\brelief\b",
        r"\bkabartma(ları|sı|sını|sında)?\b",
    ],

    "Ancient_Infrastructure": [
        r"\baqueduct\b",
    ],
    "Cave": [
        r"\bcave\b",
        r"\bmağara(sı|sını|sında|lar|ları|ların(i)?)?\b",
        r"\bobruk(ları|u|unu|unda)?\b",
    ],
    "Waterfall": [
        r"\bwaterfall\b",
        r"\bşelale(si|sini|sinde|ler|leri|lerin(i)?)?\b",
    ],
    "Canyon": [
        r"\bcanyon\b",
        r"\bkanyon(u|unu|unda|lar|ları|ların(i)?)?\b",
    ],
    "Plateau": [
        r"\bplateau\b",
        r"\byayla(sı|sını|sında|lar|ları|ların(i)?)?\b",
    ],
    "Valley": [
    r"\bvalley\b",
    r"\bvadi(si|sini|sinde|ler|leri|lerin(i)?)?\b",
    ],

    "Street": [
    r"\bstreet\b",
    r"\bsokak(ı|i|ta|te|tan|ten|lar|ları|ların|larını)?\b",
    r"\bsokağı(nı|nda|ndan)?\b",
    ],
    "Bridge": [
        r"\bbridge\b",
        r"\bköprü(sü|sünü|sünde|sünden|ler|leri|lerinin)?\b",
    ],

    "Mansion": [
        r"\bmansion\b",
        r"\bkonak(ı|lar|ları|ların|larını)?\b",
        r"\bkonağı(nı|nda|ndan)?\b",
    ],
    "Viewpoint": [
        r"\bviewpoint\b",
        r"\bpanoramic\b",
        r"\bobservation deck\b",
        r"\bseyir terası\b",
        r"\bcam teras\b",
        r"\bmanzara noktası\b",
        r"\bseyir noktası\b",
    ],
    "River": [
        r"\briver\b",
        r"\bnehir\b",
        r"\bdere(si|sini|sinde|ler|leri|lerin(i)?)?\b",
        r"\bçay(ı|ını|ında)?\b",
    ],
    "Bay": [
        r"\bbay\b",
        r"\bcove\b",
        r"\bkoy(u|unu|unda|lar|ları|ların(i)?)?\b",
    ],
    "Beach": [
        r"\bbeach\b",
        r"\bplaj(ı|ını|ında|lar|ları|ların(i)?)?\b",
        r"\bsahil\b",
        r"\bkoyu\b",
    ],
    "Island": [
        r"\bisland\b",
        r"\bada(sı|sını|sında|lar|ları|ların(i)?)?\b",
    ],
    "Lake": [
        r"\blake\b",
        r"\bgöl(ü|ünü|ünde|ler|leri|lerin(i)?)?\b",
        r"\bgölet(i|ini|inde|ler|leri|lerin(i)?)?\b",
    ],
    "Mountain": [
        r"\bmountain\b",
        r"\bdağ(ı|ını|ında|lar|ları|ların(i)?)?\b",
        r"\btepe(si|sini|sinde|ler|leri|lerin(i)?)?\b",
    ],
    "Geological_Site": [
        r"\btravertin(es)?\b",
        r"\btraverten(leri|i|ini|inde)?\b",
        r"\bjeopark\b",
        r"\bgeopark\b",
        r"\bperibacaları\b",
        r"\bfairy chimney\b",
        r"\bhoodoo(s)?\b",
        r"\bbazalt sütun(ları|u|unu|unda)?\b",
        r"\bfossil\b",
        r"\b(fosil )?lokalite(si|sini|sinde)?\b",
    ],
    "Spring": [
        r"\bspring\b",
        r"\bgöze(leri|si|sini|sinde)?\b",
    ],
    "Ski_Resort": [
        r"\bski resort\b",
        r"\bkayak merkez(i|i)?\b",
        r"\bkayak tesis(leri|i|ini|inde)?\b",
    ],
    "Thermal_Spa": [
        r"\bthermal\b",
        r"\bkaplıca(ları|sı|sını|sında)?\b",
        r"\btermal\b",
        r"\biçme(leri|si|sini|sinde)?\b",
    ],
    "Cable_Car": [
        r"\bcable car\b",
        r"\bteleferik\b",
    ],
    "Aquarium": [
        r"\baquarium\b",
        r"\bakvaryum\b",
    ],
    "Zoo": [
        r"\bzoo\b",
        r"\bhayvanat bahçe(si|sini|sinde)?\b",
        r"\bdoğal yaşam park(ı|ını|ında)?\b",
    ],
    "Recreation_Area": [
        r"\bmesire alan(ı|ını|ında)?\b",
        r"\bpiknik alan(ı|ını|ında)?\b",
        r"\bkent bahçe(si|sini|sinde)?\b",
        r"\bmillet bahçe(si|sini|sinde)?\b",
    ],
    "Dam": [
        r"\bdam\b",
        r"\bbaraj(ı|ını|ında|lar|ları|ların(i)?)?\b",
    ],
    "Aqueduct": [
        r"\baqueduct\b",
        r"\bsu kemer(i|ini|inde|ler|leri|lerin(i)?)?\b",
    ],
    "Harbor_Pier": [
        r"\bharbor\b",
        r"\bpier\b",
        r"\biskele(si|sini|sinde|ler|leri|lerin(i)?)?\b",
        r"\bliman(ı|ını|ında|lar|ları|ların(i)?)?\b",
    ],
    "Railway_Station": [
        r"\brailway station\b",
        r"\btrain station\b",
        r"\btren gar(ı|ını|ında)?\b",
        r"\bgar\b",
        r"\bistasyon(u|unu|unda)?\b",
    ],
}

# Pre-compile every pattern once so the matching loop stays cheap.
COMPILED = {
    label: [re.compile(pat, flags=re.IGNORECASE) for pat in patterns]
    for label, patterns in TYPE_PATTERNS.items()
}

# Small helpers for defensive text handling on possibly-missing fields.
def safe_text(x):
    return "" if x is None else str(x)

# Collapse whitespace/newlines so regex \b boundaries behave predictably.
def normalize(text):
    text = safe_text(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Concatenate the POI's name and all available description fields
# into a single searchable blob.
def build_text(poi):
    return normalize(" ".join([
        safe_text(poi.get("name")),
        safe_text(poi.get("wikidata_description_en")),
        safe_text(poi.get("short_description_en")),
        safe_text(poi.get("short_description_tr")),
        safe_text(poi.get("wikidata_description_tr")),
    ]))

# Return every label whose pattern set hits the text. Fall back to
# "Other" when nothing matches so every POI ends up with at least one tag.
def detect_labels(text):
    detected = []
    for label, patterns in COMPILED.items():
        for rgx in patterns:
            if rgx.search(text):
                detected.append(label)
                break
    return detected if detected else ["Other"]

# Accumulate processed POIs here before converting to a DataFrame.
rows = []

# Each file under this folder is a city's POI dump in JSON form.
folder = "cities"

# Walk through every city file and classify its POIs.
for file in os.listdir(folder):
    if not file.endswith(".json"):
        continue

    path = os.path.join(folder, file)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Some files wrap the POI list under a top-level key; unwrap if so.
    if isinstance(data, dict):
        for key in ["pois", "items", "data", "results", "components"]:
            if key in data and isinstance(data[key], list):
                data = data[key]
                break

    # Defensive skip for malformed files.
    if not isinstance(data, list):
        continue

    for poi in data:
        text = build_text(poi)
        labels = detect_labels(text)

        rows.append({
            "name": poi.get("name"),
            "lat": poi.get("lat"),
            "lon": poi.get("lon"),
            "categories": labels
        })

df = pd.DataFrame(rows)

print("\nTOTAL POI:", len(df))

# Exploding multi-label rows lets value_counts reflect true category volume.
df_exploded = df.explode("categories")

print("\n=== CATEGORY DISTRIBUTION ===")
category_counts = df_exploded["categories"].value_counts()

for cat, count in category_counts.items():
    print(f"{cat} -> {count}")

# Dump POIs that ended up with more than one label — useful for spot-checks.
multi_label_df = df[df["categories"].apply(lambda x: len(x) > 1)]

with open("multi_label_pois_v1.txt", "w", encoding="utf-8") as f:
    for _, row in multi_label_df.iterrows():
        name = safe_text(row["name"])
        cats = ", ".join(row["categories"])
        f.write(f"{name} -> {cats}\n")

print("\nMULTI-LABEL COUNT:", len(multi_label_df))
print("Saved: multi_label_pois_v1.txt")

# "Other" dump so we can inspect what the regex set missed and iterate.
other_df = df[df["categories"].apply(lambda x: x == ["Other"])]

with open("other_pois_v1.txt", "w", encoding="utf-8") as f:
    for _, row in other_df.iterrows():
        name = safe_text(row["name"])
        f.write(f"{name}\n")

print("OTHER COUNT:", len(other_df))
print("Saved: other_pois_v1.txt")

# Museum-only dump for sanity-checking the museum pattern's precision.
museum_df = df[df["categories"].apply(lambda x: "Museum" in x)]

with open("museum_pois_v1.txt", "w", encoding="utf-8") as f:
    for _, row in museum_df.iterrows():
        name = safe_text(row["name"])
        cats = ", ".join(row["categories"])
        f.write(f"{name} -> {cats}\n")

print("MUSEUM COUNT:", len(museum_df))
print("Saved: museum_pois_v1.txt")

# Serialize the label list as a comma-joined string for CSV portability.
df["categories"] = df["categories"].apply(lambda x: ", ".join(x))
df.to_csv("pois_type_based_categories_v1.csv", index=False)

print("Saved: pois_type_based_categories_v1.csv")


TOTAL POI: 2837

=== CATEGORY DISTRIBUTION ===
Museum -> 444
Ancient_City -> 439
Park -> 421
Mountain -> 401
Castle -> 236
Complex -> 191
Other -> 146
Burial_Site -> 143
Monument -> 138
Viewpoint -> 134
Mosque -> 123
Church -> 97
Lake -> 96
Tomb -> 72
Ev -> 66
Valley -> 61
Tower -> 61
Ancient_Structure -> 58
River -> 56
Köy -> 56
City_Walls_Gates -> 54
Waterfall -> 53
Cave -> 51
Mansion -> 44
Caravanserai_Inn -> 40
Monastery -> 40
Bridge -> 40
Beach -> 37
Harbor_Pier -> 37
Madrasa -> 36
Island -> 33
Square -> 32
Recreation_Area -> 31
Bath_Hammam -> 30
Canyon -> 28
Plateau -> 26
Bazaar_Market -> 23
Cemetery -> 19
Thermal_Spa -> 19
Ski_Resort -> 17
Street -> 17
Bay -> 15
Dam -> 13
Underground_City -> 10
Ancient_Relief_Art -> 10
Geological_Site -> 9
Aqueduct -> 8
Zoo -> 8
Railway_Station -> 7
Cistern -> 5
Cable_Car -> 4
Spring -> 4
Ancient_Infrastructure -> 4
Aquarium -> 2

MULTI-LABEL COUNT: 1064
Saved: multi_label_pois_v1.txt
OTHER COUNT: 146
Saved: other_pois_v1.txt
MUSEUM COUNT: 444


In [ ]:
import pandas as pd

df = pd.read_csv("pois_type_based_categories_v1.csv")

# Re-parse the comma-joined string back into a list of labels.
df["categories"] = df["categories"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

# ==================================================
# Two-level taxonomy: each granular type label maps to a
# (main_category, sub_category) tuple. Three top-level buckets:
# Museums, Cultural Heritage, Nature.
# ==================================================

MAIN_MAPPING = {

    # 🟢 1️⃣ Museums
    "Museum": ("Museums", "Museum"),

    # 🟢 2️⃣ Cultural Heritage

    # Archaeological & pre-modern sites
    "Ancient_City": ("Cultural Heritage", "Ancient & Archaeology"),
    "Burial_Site": ("Cultural Heritage", "Ancient & Archaeology"),
    "Ancient_Structure": ("Cultural Heritage", "Ancient & Archaeology"),
    "Ancient_Relief_Art": ("Cultural Heritage", "Ancient & Archaeology"),
    "Ancient_Infrastructure": ("Cultural Heritage", "Ancient & Archaeology"),
    "Underground_City": ("Cultural Heritage", "Ancient & Archaeology"),

    # Places of worship and funerary monuments
    "Mosque": ("Cultural Heritage", "Religious"),
    "Church": ("Cultural Heritage", "Religious"),
    "Monastery": ("Cultural Heritage", "Religious"),
    "Madrasa": ("Cultural Heritage", "Religious"),
    "Tomb": ("Cultural Heritage", "Religious"),
    "Cemetery": ("Cultural Heritage", "Religious"),

    # Defensive architecture
    "Castle": ("Cultural Heritage", "Fortifications"),
    "City_Walls_Gates": ("Cultural Heritage", "Fortifications"),
    "Tower": ("Cultural Heritage", "Fortifications"),

    # Vernacular and civil architecture
    "Mansion": ("Cultural Heritage", "Civil & Traditional Architecture"),
    "Caravanserai_Inn": ("Cultural Heritage", "Civil & Traditional Architecture"),
    "Bath_Hammam": ("Cultural Heritage", "Civil & Traditional Architecture"),
    "Ev": ("Cultural Heritage", "Civil & Traditional Architecture"),
    "Köy": ("Cultural Heritage", "Civil & Traditional Architecture"),
    "Bridge": ("Cultural Heritage", "Civil & Traditional Architecture"),

    # Public urban spaces and monumental heritage
    "Monument": ("Cultural Heritage", "Urban & Monumental Heritage"),
    "Square": ("Cultural Heritage", "Urban & Monumental Heritage"),
    "Bazaar_Market": ("Cultural Heritage", "Urban & Monumental Heritage"),
    "Complex": ("Cultural Heritage", "Urban & Monumental Heritage"),

    # Historic water / engineering works
    "Aqueduct": ("Cultural Heritage", "Historical Infrastructure"),
    "Cistern": ("Cultural Heritage", "Historical Infrastructure"),
    "Dam": ("Cultural Heritage", "Historical Infrastructure"),

    # Transport infrastructure treated as heritage
    "Railway_Station": ("Cultural Heritage", "Transportation as Heritage"),
    "Harbor_Pier": ("Cultural Heritage", "Transportation as Heritage"),
    "Street": ("Cultural Heritage", "Transportation as Heritage"),

    # 🟢 3️⃣ Nature

    # Geological and topographic features
    "Mountain": ("Nature", "Terrain & Landforms"),
    "Plateau": ("Nature", "Terrain & Landforms"),
    "Valley": ("Nature", "Terrain & Landforms"),
    "Canyon": ("Nature", "Terrain & Landforms"),
    "Cave": ("Nature", "Terrain & Landforms"),
    "Geological_Site": ("Nature", "Terrain & Landforms"),

    # Water bodies and coastline
    "Lake": ("Nature", "Water & Coastal"),
    "River": ("Nature", "Water & Coastal"),
    "Bay": ("Nature", "Water & Coastal"),
    "Beach": ("Nature", "Water & Coastal"),
    "Island": ("Nature", "Water & Coastal"),
    "Spring": ("Nature", "Water & Coastal"),
    "Waterfall": ("Nature", "Water & Coastal"),

    # Recreational outdoor attractions
    "Park": ("Nature", "Parks & Outdoor"),
    "Recreation_Area": ("Nature", "Parks & Outdoor"),
    "Viewpoint": ("Nature", "Parks & Outdoor"),
    "Ski_Resort": ("Nature", "Parks & Outdoor"),
    "Thermal_Spa": ("Nature", "Parks & Outdoor"),
    "Cable_Car": ("Nature", "Parks & Outdoor"),

    # Fauna-oriented attractions
    "Zoo": ("Nature", "Wildlife & Natural Experience"),
    "Aquarium": ("Nature", "Wildlife & Natural Experience"),
}

# ==================================================
# Projecting a POI's list of type labels into unique
# sets of (main, sub) categories. Sets deduplicate
# cases where several labels share the same bucket.
# ==================================================

def assign_multi_main_sub(label_list):

    main_set = set()
    sub_set = set()

    for label in label_list:
        if label in MAIN_MAPPING:
            main, sub = MAIN_MAPPING[label]
            main_set.add(main)
            sub_set.add(sub)

    # If nothing mapped (only "Other" etc.), mark the POI as Other for review.
    if not main_set:
        return ["Other"], ["Other"]

    return list(main_set), list(sub_set)

df[["main_category", "sub_category"]] = df["categories"].apply(
    lambda x: pd.Series(assign_multi_main_sub(x))
)

df["main_category"] = df["main_category"].apply(lambda x: ", ".join(x))
df["sub_category"] = df["sub_category"].apply(lambda x: ", ".join(x))

# ==================================================
# Sanity-check distributions after the mapping.
# ==================================================

print("\n=== MAIN CATEGORY DISTRIBUTION ===")
print(df["main_category"].str.split(", ").explode().value_counts())

print("\n=== SUB CATEGORY DISTRIBUTION ===")
print(df["sub_category"].str.split(", ").explode().value_counts())

df.to_csv("pois_with_main_sub_multi_v1.csv", index=False)

print("\nSaved: pois_with_main_sub_multi_v1.csv")


=== MAIN CATEGORY DISTRIBUTION ===
main_category
Cultural Heritage    1597
Nature               1160
Museums               444
Other                 146
Name: count, dtype: int64

=== SUB CATEGORY DISTRIBUTION ===
sub_category
Parks & Outdoor                     592
Ancient & Archaeology               558
Terrain & Landforms                 549
Museum                              444
Urban & Monumental Heritage         374
Religious                           344
Fortifications                      321
Water & Coastal                     275
Civil & Traditional Architecture    263
Other                               146
Transportation as Heritage           59
Historical Infrastructure            26
Wildlife & Natural Experience        10
Name: count, dtype: int64

Saved: pois_with_main_sub_multi_v1.csv


In [ ]:
import pandas as pd

# Load the v1 main/sub-assigned dataset.
df = pd.read_csv("pois_with_main_sub_multi_v1.csv")

# Expand comma-joined category strings back into lists for counting.
df["main_category_list"] = df["main_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

df["sub_category_list"] = df["sub_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

# ==================================================
# QA dump #1: POIs that landed in more than two main categories.
# Anything over 2 is likely an over-tagging artifact and needs review.
# ==================================================

main_over_2 = df[df["main_category_list"].apply(lambda x: len(x) > 2)]

with open("pois_more_than_2_main_categories.txt", "w", encoding="utf-8") as f:
    for _, row in main_over_2.iterrows():
        f.write(f"{row['name']} -> {', '.join(row['main_category_list'])}\n")

print("POIs with >2 main categories:", len(main_over_2))
print("Saved: pois_more_than_2_main_categories.txt")


# ==================================================
# QA dump #2: POIs with more than four sub-categories.
# Same rationale: the downstream schema caps sub slots at 4.
# ==================================================

sub_over_4 = df[df["sub_category_list"].apply(lambda x: len(x) > 4)]

with open("pois_more_than_4_sub_categories.txt", "w", encoding="utf-8") as f:
    for _, row in sub_over_4.iterrows():
        f.write(f"{row['name']} -> {', '.join(row['sub_category_list'])}\n")

print("POIs with >4 sub categories:", len(sub_over_4))
print("Saved: pois_more_than_4_sub_categories.txt")

POIs with >2 main categories: 8
Saved: pois_more_than_2_main_categories.txt
POIs with >4 sub categories: 6
Saved: pois_more_than_4_sub_categories.txt


In [ ]:
import pandas as pd

df = pd.read_csv("pois_with_main_sub_multi_v1.csv")

# Hand-curated overrides for POIs the regex pipeline miscategorised.
# Each entry forces the POI's main categories to the given list; its
# sub-categories are then filtered to those compatible with the new mains.
manual_override = {
    "Harput Kalesi Fincan Müzesi": ["Cultural Heritage", "Museums"],
    "Kedrai Nekropolü": ["Cultural Heritage"],
    "Somuncu Baba Külliyesi": ["Cultural Heritage"],
    "Dam Dağı": ["Nature"],
    "İda Madra Jeopark Müzesi": ["Nature", "Museums"],
    "Bozcaada Yerel Tarih Araştırma Müzesi": ["Museums"],
    "Tekkeköy Mağaraları": ["Nature"],
    "Karanlık kilise": ["Cultural Heritage"],
}

# Allowed sub-categories under each main — used to strip any sub that no
# longer belongs after the main category gets overwritten.
MAIN_TO_SUB = {
    "Museums": ["Museum"],
    "Cultural Heritage": [
        "Ancient & Archaeology",
        "Religious",
        "Fortifications",
        "Civil & Traditional Architecture",
        "Urban & Monumental Heritage",
        "Historical Infrastructure",
        "Transportation as Heritage",
    ],
    "Nature": [
        "Terrain & Landforms",
        "Water & Coastal",
        "Parks & Outdoor",
        "Wildlife & Natural Experience",
    ],
}

df["main_category_list"] = df["main_category"].str.split(", ")
df["sub_category_list"] = df["sub_category"].str.split(", ")

for poi_name, new_mains in manual_override.items():

    idx = df.index[df["name"] == poi_name]

    # Tolerate POI names that may have been filtered out upstream.
    if len(idx) == 0:
        continue

    i = idx[0]

    # Replace the main categories outright.
    df.at[i, "main_category_list"] = new_mains

    # Rebuild the allowed sub-category whitelist for this POI's new mains.
    allowed_subs = []
    for m in new_mains:
        allowed_subs.extend(MAIN_TO_SUB.get(m, []))

    # Drop subs that don't belong under any of the new mains.
    df.at[i, "sub_category_list"] = [
        s for s in df.at[i, "sub_category_list"] if s in allowed_subs
    ]

# Re-serialize lists back to comma-joined strings for CSV output.
df["main_category"] = df["main_category_list"].apply(lambda x: ", ".join(x))
df["sub_category"] = df["sub_category_list"].apply(lambda x: ", ".join(x))

df.drop(columns=["main_category_list", "sub_category_list"], inplace=True)

df.to_csv("pois_with_main_sub_multi_v2.csv", index=False)

print("Updated file saved as pois_with_main_sub_multi_v2.csv")

Updated file saved as pois_with_main_sub_multi_v2.csv


In [ ]:
import pandas as pd

# Re-run the same QA check after the v2 main-category overrides.
df = pd.read_csv("pois_with_main_sub_multi_v2.csv")

df["main_category_list"] = df["main_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

df["sub_category_list"] = df["sub_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

# ==================================================
# Still checking: POIs with more than 2 main categories.
# ==================================================

main_over_2 = df[df["main_category_list"].apply(lambda x: len(x) > 2)]

with open("pois_more_than_2_main_categories.txt", "w", encoding="utf-8") as f:
    for _, row in main_over_2.iterrows():
        f.write(f"{row['name']} -> {', '.join(row['main_category_list'])}\n")

print("POIs with >2 main categories:", len(main_over_2))
print("Saved: pois_more_than_2_main_categories.txt")


# ==================================================
# Still checking: POIs with more than 4 sub-categories.
# These are what cell 6 will target with its manual sub override.
# ==================================================

sub_over_4 = df[df["sub_category_list"].apply(lambda x: len(x) > 4)]

with open("pois_more_than_4_sub_categories.txt", "w", encoding="utf-8") as f:
    for _, row in sub_over_4.iterrows():
        f.write(f"{row['name']} -> {', '.join(row['sub_category_list'])}\n")

print("POIs with >4 sub categories:", len(sub_over_4))
print("Saved: pois_more_than_4_sub_categories.txt")

POIs with >2 main categories: 0
Saved: pois_more_than_2_main_categories.txt
POIs with >4 sub categories: 4
Saved: pois_more_than_4_sub_categories.txt


In [ ]:
import pandas as pd

df = pd.read_csv("pois_with_main_sub_multi_v2.csv")

# POIs whose sub-category list exceeded the 4-slot budget get an explicit
# curated sub list here. Their main categories are then reconstructed
# from these subs so everything stays consistent.
manual_sub_override = {

    "Şar Dağı": [
        "Ancient & Archaeology",
        "Terrain & Landforms",
    ],

    "Hunat Hatun Külliyesi": [
        "Museum",
        "Religious",
        "Civil & Traditional Architecture",
    ],

    "Giresun Kalesi": [
        "Fortifications",
        "Ancient & Archaeology",
        "Water & Coastal",
    ],

    "Phaselis Örenyeri": [
        "Ancient & Archaeology",
        "Water & Coastal",
        "Historical Infrastructure",
    ],
}

# Reverse of MAIN_TO_SUB: each sub-category maps back to its parent main.
# Used to rebuild the main list from the curated sub list.
SUB_TO_MAIN = {
    # Museums
    "Museum": "Museums",

    # Cultural Heritage
    "Ancient & Archaeology": "Cultural Heritage",
    "Religious": "Cultural Heritage",
    "Fortifications": "Cultural Heritage",
    "Civil & Traditional Architecture": "Cultural Heritage",
    "Urban & Monumental Heritage": "Cultural Heritage",
    "Historical Infrastructure": "Cultural Heritage",
    "Transportation as Heritage": "Cultural Heritage",

    # Nature
    "Terrain & Landforms": "Nature",
    "Water & Coastal": "Nature",
    "Parks & Outdoor": "Nature",
    "Wildlife & Natural Experience": "Nature",
}

df["main_category_list"] = df["main_category"].str.split(", ")
df["sub_category_list"] = df["sub_category"].str.split(", ")

for poi_name, new_subs in manual_sub_override.items():

    idx = df.index[df["name"] == poi_name]

    if len(idx) == 0:
        continue

    i = idx[0]

    # Overwrite the sub-category list with the curated one.
    df.at[i, "sub_category_list"] = new_subs

    # Derive the corresponding main categories from the new subs.
    new_mains = set()
    for sub in new_subs:
        if sub in SUB_TO_MAIN:
            new_mains.add(SUB_TO_MAIN[sub])

    df.at[i, "main_category_list"] = list(new_mains)

df["main_category"] = df["main_category_list"].apply(lambda x: ", ".join(x))
df["sub_category"] = df["sub_category_list"].apply(lambda x: ", ".join(x))

df.drop(columns=["main_category_list", "sub_category_list"], inplace=True)

df.to_csv("pois_with_main_sub_multi_v3.csv", index=False)

print("Saved: pois_with_main_sub_multi_v3.csv")

Saved: pois_with_main_sub_multi_v3.csv


In [ ]:
import pandas as pd

# Final sanity check on v3, now that both main and sub overrides have run.
df = pd.read_csv("pois_with_main_sub_multi_v3.csv")

df["main_category_list"] = df["main_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

df["sub_category_list"] = df["sub_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

# ==================================================
# Anything still over the main-count budget at this stage is a bug.
# ==================================================

main_over_2 = df[df["main_category_list"].apply(lambda x: len(x) > 2)]

with open("pois_more_than_2_main_categories.txt", "w", encoding="utf-8") as f:
    for _, row in main_over_2.iterrows():
        f.write(f"{row['name']} -> {', '.join(row['main_category_list'])}\n")

print("POIs with >2 main categories:", len(main_over_2))
print("Saved: pois_more_than_2_main_categories.txt")


# ==================================================
# Likewise for sub-category over-count.
# ==================================================

sub_over_4 = df[df["sub_category_list"].apply(lambda x: len(x) > 4)]

with open("pois_more_than_4_sub_categories.txt", "w", encoding="utf-8") as f:
    for _, row in sub_over_4.iterrows():
        f.write(f"{row['name']} -> {', '.join(row['sub_category_list'])}\n")

print("POIs with >4 sub categories:", len(sub_over_4))
print("Saved: pois_more_than_4_sub_categories.txt")

POIs with >2 main categories: 0
Saved: pois_more_than_2_main_categories.txt
POIs with >4 sub categories: 0
Saved: pois_more_than_4_sub_categories.txt


In [ ]:
import pandas as pd

# Load the v3 dataset, which is now clean at the list level.
df = pd.read_csv("pois_with_main_sub_multi_v3.csv")

# Re-parse category strings into lists for positional slicing.
df["main_list"] = df["main_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

df["sub_list"] = df["sub_category"].fillna("").apply(
    lambda x: [c.strip() for c in x.split(",") if c.strip()]
)

# Flatten into two fixed main-category columns (max 2 mains per POI).
df["main_1"] = df["main_list"].apply(lambda x: x[0] if len(x) > 0 else "")
df["main_2"] = df["main_list"].apply(lambda x: x[1] if len(x) > 1 else "")

# Flatten into four fixed sub-category columns (max 4 subs per POI).
df["sub_1"] = df["sub_list"].apply(lambda x: x[0] if len(x) > 0 else "")
df["sub_2"] = df["sub_list"].apply(lambda x: x[1] if len(x) > 1 else "")
df["sub_3"] = df["sub_list"].apply(lambda x: x[2] if len(x) > 2 else "")
df["sub_4"] = df["sub_list"].apply(lambda x: x[3] if len(x) > 3 else "")

# Drop the intermediate string/list columns — the flat columns are the schema.
df.drop(columns=["main_category", "sub_category", "main_list", "sub_list"], inplace=True)

df.to_csv("pois_with_main_sub_multi_v4.csv", index=False)

print("Saved: pois_with_main_sub_multi_v4.csv")

Saved: pois_with_main_sub_multi_v4.csv


In [ ]:
import pandas as pd

df = pd.read_csv("pois_with_main_sub_multi_v4.csv")

sub_cols = ["sub_1", "sub_2", "sub_3", "sub_4"]

# Authoritative whitelist: which sub-categories are valid under each main.
# Used to filter out cross-bucket contamination when reporting counts.
MAIN_TO_ALLOWED_SUBS = {
    "Museums": ["Museum"],
    "Cultural Heritage": [
        "Ancient & Archaeology",
        "Religious",
        "Fortifications",
        "Civil & Traditional Architecture",
        "Urban & Monumental Heritage",
        "Historical Infrastructure",
        "Transportation as Heritage",
    ],
    "Nature": [
        "Terrain & Landforms",
        "Water & Coastal",
        "Parks & Outdoor",
        "Wildlife & Natural Experience",
    ],
}

print("\n========== CATEGORY DISTRIBUTION ==========\n")

# Walk each main bucket, count POIs in it, then list its sub breakdown.
for main, allowed_subs in MAIN_TO_ALLOWED_SUBS.items():

    # A POI is in this main if either main_1 or main_2 matches.
    main_mask = (df["main_1"] == main) | (df["main_2"] == main)
    main_count = main_mask.sum()

    print(f"\n🟢 {main} ({main_count})")

    if main_count == 0:
        continue

    df_main = df[main_mask]

    # Stack all sub slots for this main's POIs into a single series.
    sub_series = pd.concat([df_main[col] for col in sub_cols])
    sub_series = sub_series.dropna()
    sub_series = sub_series[sub_series != ""]

    # Only report subs that legitimately belong under this main.
    sub_series = sub_series[sub_series.isin(allowed_subs)]

    sub_counts = sub_series.value_counts().sort_index()

    for sub in sub_counts.index:
        print(f"   └── {sub} ({sub_counts[sub]})")

# Report any leftover "Other" POIs — these are the ones Notebook 2 will
# manually label or delete.
other_mask = (df["main_1"] == "Other") | (df["main_2"] == "Other")
print(f"\n🔴 Other ({other_mask.sum()})")


========== CATEGORY DISTRIBUTION ==========


🟢 Museums (439)
   └── Museum (439)

🟢 Cultural Heritage (1593)
   └── Ancient & Archaeology (557)
   └── Civil & Traditional Architecture (259)
   └── Fortifications (319)
   └── Historical Infrastructure (25)
   └── Religious (341)
   └── Transportation as Heritage (58)
   └── Urban & Monumental Heritage (370)

🟢 Nature (1155)
   └── Parks & Outdoor (592)
   └── Terrain & Landforms (545)
   └── Water & Coastal (273)
   └── Wildlife & Natural Experience (10)

🔴 Other (146)
